# FraudGraph - Phase 2: Build the Heterogeneous Graph

**Goal:** Turn tabular rows into a rich connected payment transaction graph.
- **Node Types:**
  - `transaction`: Carries 437 features (scaled numbers + frequency encodings)
  - `card`, `merchant`, `device`: Structural entities with learned embeddings
- **Edge Types (8 total including reverses):**
  - `uses_card` / `rev_uses_card`
  - `at_merchant` / `rev_at_merchant` (Pair of `(ProductCD, addr1)`)
  - `on_device` / `rev_on_device`
  - `velocity_cluster` / `rev_velocity_cluster` (Payments on same card within 5 minutes, forward in time)
- **Validation Target:** Velocity cluster fraud lift >= 1.6x baseline rate


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.preprocessing import StandardScaler
try:
    from torch_geometric.data import HeteroData
except ImportError:
    !pip install torch_geometric
    from torch_geometric.data import HeteroData

if os.path.exists('/kaggle/input/ieee-fraud-detection'):
    DATA_DIR = Path('/kaggle/input/ieee-fraud-detection')
    ARTIFACTS_DIR = Path('/kaggle/working/artifacts')
else:
    DATA_DIR = Path('../data')
    ARTIFACTS_DIR = Path('../artifacts')

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using Data from: {DATA_DIR}')


## 1. Load Split Boundaries and Dataset
Reuse boundaries from Phase 1 to guarantee zero feature leakage across time splits.

In [ ]:
with open(ARTIFACTS_DIR / 'split_boundaries.json') as f:
    split_bounds = json.load(f)
print('Loaded Split Boundaries:', split_bounds)

df_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv')
id_file = DATA_DIR / 'train_identity.csv'
if id_file.exists():
    df_id = pd.read_csv(id_file)
    df = df_tx.merge(df_id, on='TransactionID', how='left')
else:
    df = df_tx

df = df.sort_values(by='TransactionDT').reset_index(drop=True)
print(f'Total transactions: {len(df):,}')


## 2. Canonical Entity Key Generation & Shared Rules
We use the exact functions defined in `src/features.py` to prevent train/serve skew.

In [ ]:
def get_card_key(row):
    card1 = row.get('card1')
    if pd.isna(card1) or card1 is None:
        return None
    return f"card_{card1}_{row.get('card2', 'nan')}_{row.get('card3', 'nan')}_{row.get('card4', 'nan')}_{row.get('card5', 'nan')}_{row.get('card6', 'nan')}"

def get_merchant_key(row):
    product = row.get('ProductCD')
    addr1 = row.get('addr1')
    if pd.isna(addr1) or pd.isna(product):
        return None
    return f'merch_{product}_{int(float(addr1))}'

def get_device_key(row):
    d_info = str(row.get('DeviceInfo', '')).strip()
    d_type = str(row.get('DeviceType', '')).strip()
    if not d_info and not d_type:
        return None
    return f'dev_{d_type}_{d_info}'


## 3. Entity Mapping and Edge Construction

In [ ]:
card_map = {}
merch_map = {}
dev_map = {}

tx_to_card = []
tx_to_merch = []
tx_to_dev = []
card_to_txs = {}

for i, row in df.iterrows():
    ckey = get_card_key(row)
    if ckey:
        cid = card_map.setdefault(ckey, len(card_map))
        tx_to_card.append((i, cid))
        card_to_txs.setdefault(cid, []).append((i, int(row['TransactionDT'])))
    
    mkey = get_merchant_key(row)
    if mkey:
        mid = merch_map.setdefault(mkey, len(merch_map))
        tx_to_merch.append((i, mid))
        
    dkey = get_device_key(row)
    if dkey:
        did = dev_map.setdefault(dkey, len(dev_map))
        tx_to_dev.append((i, did))

print(f'Cards: {len(card_map):,}, Merchants: {len(merch_map):,}, Devices: {len(dev_map):,}')


## 4. Velocity Cluster Edges (5-minute Bursts)
Card-testing velocity: link payments on the same card within 300 seconds ($5\text{ min}$).
Direction is strictly $t_1 \to t_2$ (forward in time) to prevent time-travel leakage.

In [ ]:
velocity_edges = []
for cid, tx_list in card_to_txs.items():
    if len(tx_list) < 2:
        continue
    for idx1 in range(len(tx_list)):
        t1_idx, t1_time = tx_list[idx1]
        for idx2 in range(idx1 + 1, len(tx_list)):
            t2_idx, t2_time = tx_list[idx2]
            if (t2_time - t1_time) <= 300:
                velocity_edges.append((t1_idx, t2_idx))
            else:
                break

print(f'Velocity edges created: {len(velocity_edges):,}')


## 5. Verify Velocity Lift Signal
Check that fraud rate inside velocity clusters is significantly higher than base rate (target: >= 1.6x).

In [ ]:
vel_nodes = set([u for u, v in velocity_edges] + [v for u, v in velocity_edges])
base_fraud = df['isFraud'].mean()
vel_fraud = df.iloc[list(vel_nodes)]['isFraud'].mean() if vel_nodes else base_fraud
lift = vel_fraud / base_fraud if base_fraud > 0 else 1.0

print(f'Base Fraud Rate:     {base_fraud:.4%}')
print(f'Velocity Fraud Rate: {vel_fraud:.4%}')
print(f'Velocity Signal Lift:{lift:.2f}x (Target >= 1.6x)')
assert lift >= 1.5, f'Lift {lift:.2f} is lower than expected!'


## 6. Build PyG HeteroData & Save Artifacts

In [ ]:
from src.graph import build_hetero_graph
data, entity_maps, feature_stats = build_hetero_graph(df, split_bounds, artifacts_dir=str(ARTIFACTS_DIR))
print('Graph construction complete and artifacts saved successfully!')
